# `C00` — Inertness and the contract-status gate

Scenario **C00** of the Track A certification ladder (card `OMIE-CERT-C00-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C0). Oracle kind: **exact-literal**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; every run is done both ways and the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. *Structural equality* means two state trees equal leaf for leaf.

## 1. Purpose

**Purpose.** Before any claim about what the engine *does*, establish what it does when it should do
nothing. Three things are pinned. First, that an empty book is genuinely empty and a fresh state is
reproducible leaf-for-leaf. Second — and this is the one that matters downstream — that the **pad row
sentinel is truly inert**: every tape is right-padded to a common length, and if a pad row ever
submitted an order or advanced the clock, every episode length and every reward in Track B would be
quietly wrong with no visible symptom. Third, that the contract-status gate refuses orders outside
the trading window.

The third claim carries a wrinkle worth certifying honestly. The code gates on a single
`status == OPEN` test, collapsing PENDING, SUSPENDED and CLOSED into one rejection branch. That is
conformant for submission — §3.3 suspends order submission in the IDA clearing window — but it cannot
express the rest of §3.3's phase 2: resting orders survive the window and are batch-matched at reopen
(v2.10, OQ-28), whereas at CLOSED they are cancelled. PENDING and SUSPENDED are **simulator statuses**
with no market-spec existence as contract statuses: §9.2's contract status is OPEN / CLOSED, and
suspension is its separate "IDA suspension phase" variable. The scenario therefore exercises all three
non-OPEN statuses and certifies the collapse as **observed behaviour**. Note also that pad-row inertness has **no spec section at all** — it is a
simulator artefact, and saying so is more honest than laundering it under a lifecycle citation.

**Certifies** (card): §3.1's gate rule for a contract's **given** GCT (the status gate) — the v2.10 hour-aligned GCT(k) schedule is quoted as context, not certified; §5.4 bullet 4; §9.2 row "Contract status". Pad-row inertness: **no spec section — simulator artefact.**

**Does not certify.** §3.3 (v2.10) survival of resting orders through the IDA clearing window and the reopen batch match (OQ-28): no engine path exists, and nothing rests in this scenario's gated runs. GCT expiry of resting orders at CLOSED is C11's claim. The hour-aligned GCT(k) schedule (§3.1 v2.10) is a loader/environment property (OQ-27), not exercised here.

## 2. The rule, quoted

The governing passages, quoted **verbatim** and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`.

How the quotes ground the expectation:

- **The gate.** §3.1 and §5.4 refuse orders after GCT (CLOSED); §3.3 phase 2 suspends order submission and matching in the IDA clearing window (the simulator's SUSPENDED). Before GOT (the simulator's PENDING) the contract is not yet tradable. In all three, a submitted order must be rejected and must leave the book untouched.
- **Why the clock must not move on a rejection.** §5.2: an order receives a timestamp *when entering the SOB*. A rejected order never enters it, so the engine's timestamp counter must be exactly what it was.
- **Statuses.** §9.2's contract status is OPEN / CLOSED only; suspension is the separate "IDA suspension phase" variable. PENDING and SUSPENDED are therefore **simulator statuses**, and the single `!= OPEN` rejection they share with CLOSED is certified as **observed behaviour** (domain review v3.0, V3-12). It is conformant for submission; it cannot express the survival of resting orders through the window (quoted below as context, not certified).
- **§3.3's Matching Algorithm §5.5 source sentence is deliberately not quoted**: it concerns capacity-update batch matching (domain review v3.0, ruling R-1, quoted below). The suspension rests on OMIE Rules §58.2 (e)–(f) and Instruction 1/2024 §8, shown as citations (regulatory corpus; not machine-checked). The v2.10 reopen citation (§5.5 with §6.3) is apt for the reopen batch match (R-1 point 3), which C00 does not exercise.
- **Pad-row inertness has no spec section at all.** It is a simulator artefact: every tape is right-padded with the loader's sentinel row (`order_id = -1`, `timestamp = -1`, shown in section 4). Its expectation (a pad row submits nothing, changes nothing and advances no clock) comes from the scenario's purpose, not from `market-spec.md`, and the notebook says so rather than citing a lifecycle rule for it.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- certified: the status gate -------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§3.1",
        "**Per-contract gate closure (GCT):** 1 hour before the start of the delivery period. "
        "Orders on a contract are automatically cancelled at GCT if unmatched.",
        note="The gate rule for a contract's given GCT. Past GCT the contract is CLOSED.",
    ),
    SpecQuote(
        "market-spec.md", "§5.4",
        "Order submitted after GCT for the target contract.",
        note="§5.4 bullet 4: a rejection condition. A rejected order never enters the book.",
    ),
    SpecQuote(
        "market-spec.md", "§3.3",
        "**Phase 2 — IDA clearing window (GCT to results, ~20 min):**",
        note="The phase the simulator's SUSPENDED status stands for.",
    ),
    SpecQuote(
        "market-spec.md", "§3.3",
        "**Both regular matching and explicit allocation are fully suspended.**",
        note="Quoted without its Matching Algorithm §5.5 source sentence (domain review v3.0, R-1).",
    ),
    SpecQuote(
        "market-spec.md", "§3.3",
        "**Order submission is also suspended** for affected contracts.",
        note="Submission to a SUSPENDED contract must be refused.",
    ),
    SpecQuote(
        "market-spec.md", "§3.3",
        "Model the GCT-to-results window (20 minutes) as a full suspension of both matching and "
        "order submission for the affected delivery-day contracts.",
        note="The spec's own instruction for the simulator.",
    ),
    SpecQuote(
        "market-spec.md", "§9.2",
        "| Contract status | OPEN / CLOSED (past GCT) |",
        note="The market-spec contract status is OPEN / CLOSED only: PENDING and SUSPENDED are simulator statuses.",
    ),
    SpecQuote(
        "market-spec.md", "§9.2",
        "| IDA suspension phase | NORMAL / CAPACITY_HALT (pre-IDA) / FULL_SUSPENSION (IDA clearing) |",
        note="Suspension is a separate variable in the spec, not a contract status (V3-12).",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "Each order receives a timestamp when entering the SOB.",
        note="A rejected order never enters the SOB, so a rejection must not advance the engine's timestamp counter.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "R-1",
        "Domain review v3.0 R-1: this source sentence concerns capacity-update batch matching; "
        "the IDA-window suspension rests on OMIE Rules §58.2 (e)–(f) and Instruction 1/2024 §8.",
        note="Why §3.3's Matching Algorithm §5.5 source sentence is not quoted above.",
    ),
    # --- context only: quoted, not certified ----------------------------------------------
    SpecQuote(
        "market-spec.md", "§3.1",
        "In the Iberian continuous market, GCT is aligned to the **delivery hour**, not to each quarter-hour.",
        note="CONTEXT, not certified: the v2.10 hour-aligned GCT(k) schedule. The engine takes GCT as "
             "a given per-contract input; no Track A scenario derives it from the slot (V3-13).",
    ),
    SpecQuote(
        "market-spec.md", "§3.3",
        "It does not cancel orders resting in the continuous book for contracts that stay open, "
        "so those orders survive the gap.",
        note="CONTEXT, not certified: the part of §3.3 phase 2 the single != OPEN rejection cannot "
             "express (resting orders survive SUSPENDED; at CLOSED they are cancelled). No engine path.",
    ),
)
CITATIONS = (
    Citation(
        "OMIE Day-Ahead and Intraday Electricity Market Rules §58.2 (e)",
        "Likewise, bid submission on the continuous market shall be stopped from the closing of the "
        "bid submission period for each intraday session and until the end of the intraday auction, "
        "estimated in a normal situation as twenty (20) minutes after the closing of the bid "
        "submission period for the auction.",
    ),
    Citation(
        "OMIE Day-Ahead and Intraday Electricity Market Rules §58.2 (f)",
        "Once this process has been completed, and in accordance with the times described in Annex 1, "
        "the continuous market contracts that were suspended from trading because of the intraday "
        "auction session shall be re-opened for trading.",
    ),
    Citation(
        "OMIE Instruction 1/2024 §8",
        "Continuous allocation (and order submission) will continue with Round 1st (14/06/2024) at "
        "22:20h (13/06/2024) after the successful completion of IDA2.",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"quotes: {sum(c.ok for c in QUOTE_CHECKS)}/{len(QUOTE_CHECKS)} match their source")


## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

**Why `DRAWS = 8`, not the 500 of a random scenario.** C00's dataset is deterministic (card: *"Deterministic: an empty tape; 64 pure pad rows; a valid random tape submitted to each of PENDING, SUSPENDED and CLOSED"*) and its oracle is exact-literal: the expected record does not depend on the tape's prices or sizes. Eight seeds (0–7, the generator's own digest seeds) vary the incidental constants of the gated tape (side, price, sizes) so that a gate which happened to depend on one of them is still caught, at a cost well inside the ~40 s forecast.

**What a "draw" counts here.** The results JSON counts **records**: the two seedless records (empty tape, pad-only tape) plus four per seed (PENDING, SUSPENDED, CLOSED and the OPEN non-vacuity control), so `requested = 2 + 4 x DRAWS`. A seed whose OPEN control breaches the capacity envelope (I8 `invalid`) is discarded as a whole and the next seed is drawn; a two-order tape cannot breach it, so the discard rate is expected to be 0.

In [ ]:
CERT_ID = "C00"   # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 8         # seeds of the status-gate tape (deterministic scenario; see the note above)
SEED0 = 0         # first seed


In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.contract import ContractStatus
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig

C_EMPTY, C_PAD, C_GATE = "C00.empty", "C00.pad_only", "C00.status_gate"
CONSTRUCTIONS = (C_EMPTY, C_PAD, C_GATE)
CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE

# The three non-OPEN statuses, each exercised; OPEN is the non-vacuity control only.
STATUSES = (ContractStatus.PENDING, ContractStatus.SUSPENDED, ContractStatus.CLOSED)
OPEN = ContractStatus.OPEN

MAX_ATTEMPTS = 2 * DRAWS            # the seed loop stops at DRAWS valid seeds or after 2 x DRAWS attempts
REQUESTED_RECORDS = 2 + 4 * DRAWS   # empty + pad-only + (3 gated + 1 OPEN control) per seed

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C00_inertness_and_status.ipynb"

print(f"{CERT_ID}: {', '.join(CONSTRUCTIONS)}; {DRAWS} gate seeds from {SEED0} "
      f"(at most {MAX_ATTEMPTS} attempts); {REQUESTED_RECORDS} records requested")
print("gated statuses:", ", ".join(f"{s.name} ({int(s)})" for s in STATUSES), f"| control: {OPEN.name} ({int(OPEN)})")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW")
print(f"capacity envelope: {ENVELOPE}")


## 4. Dataset

Three constructions from the generator (`omie_tape_synth.CONSTRUCTIONS`):

- `C00.empty` — a `(0, 13)` tape: no rows at all.
- `C00.pad_only` — 64 rows, every one the loader's pad sentinel `PAD_ORDER_ROW` (`order_id = -1`, `timestamp = -1`).
- `C00.status_gate` — for each seed, a valid two-order crossing tape (one resting NON order and one NON order priced through it), right-padded with the sentinel. It is submitted to a contract in each of PENDING, SUSPENDED and CLOSED, and once to an OPEN contract as the control.

Below: the pad sentinel column by column, then the first seed's tape in readable units beside the raw integers, every seed's parameters, and the book shape the gate must refuse.

In [ ]:
from omiecid_simulator.jaxomiecid.omie_data_loader import PAD_ORDER_ROW
from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

COLS = ("contract_id", "side", "price_int", "qty_int", "order_id", "agent_id", "timestamp",
        "exec_restriction", "validity", "expiry_time", "status", "peak_qty_int", "ppd_int")

DRAW_EMPTY = synth.CONSTRUCTIONS[C_EMPTY](None)
DRAW_PAD = synth.CONSTRUCTIONS[C_PAD](None)
DRAWN = {s: synth.CONSTRUCTIONS[C_GATE](s) for s in range(SEED0, SEED0 + DRAWS)}
first = DRAWN[SEED0]

print(f"{C_EMPTY}: tape shape {DRAW_EMPTY.tape.shape}, real rows {DRAW_EMPTY.n_real_rows()}")
print(f"{C_PAD}: tape shape {DRAW_PAD.tape.shape}, real rows {DRAW_PAD.n_real_rows()}, "
      f"params {DRAW_PAD.params}")
print(f"every pad-only row equals PAD_ORDER_ROW: "
      f"{bool((DRAW_PAD.tape == np.asarray(PAD_ORDER_ROW, dtype=np.int32)).all())}")
display(Markdown("**The pad sentinel `PAD_ORDER_ROW`** (a simulator artefact; no spec section)\n\n"
                 "| " + " | ".join(COLS) + " |\n|" + "---|" * len(COLS) + "\n| "
                 + " | ".join(str(v) for v in PAD_ORDER_ROW) + " |"))


def tape_table(draw):
    rows = ["| oid | side | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | ts |",
            "|---|---|---|---|---|---|---|---|"]
    for r in draw.tape[draw.tape[:, synth.COL_OID] >= 1]:
        rows.append(
            f"| {int(r[synth.COL_OID])} | {'BUY' if int(r[synth.COL_SIDE]) == 0 else 'SELL'} | "
            f"{decode_price(int(r[synth.COL_PRICE]), CONFIG):.2f} | {decode_qty(int(r[synth.COL_QTY]), CONFIG):.1f} | "
            f"{int(r[synth.COL_PRICE])} | {int(r[synth.COL_QTY])} | {int(r[synth.COL_AGENT])} | "
            f"{int(r[synth.COL_TS])} |")
    n_pad = int(np.sum(draw.tape[:, synth.COL_OID] <= 0))
    return "\n".join(rows) + f"\n\n(+ {n_pad} pad rows)"


display(Markdown(f"**{C_GATE}, seed {SEED0}** (digest `{first.digest()[:16]}…`)\n\n" + tape_table(first)))
for s, d in DRAWN.items():
    p = d.params
    print(f"seed {s}: rest {'BUY' if p['rest_side'] == 0 else 'SELL'} {p['p_rest_eur']:.2f} EUR/MWh x "
          f"{p['q_rest_mw']:.1f} MW (int {p['p_rest_int']} / {p['q_rest_int']}), aggressor "
          f"{p['p_agg_eur']:.2f} x {p['q_agg_mw']:.1f} (gap {p['gap_ticks']} ticks); "
          f"generator statuses {p['contract_statuses']}; real rows {d.n_real_rows()}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2.6))
for r in first.tape[first.tape[:, synth.COL_OID] >= 1]:
    buy = int(r[synth.COL_SIDE]) == 0
    ax.barh(decode_price(int(r[synth.COL_PRICE]), CONFIG), decode_qty(int(r[synth.COL_QTY]), CONFIG) * (1 if buy else -1),
            height=0.6, color="tab:green" if buy else "tab:red", label="BUY" if buy else "SELL")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("quantity (MW): buy to the right, sell to the left")
ax.set_ylabel("limit price (EUR/MWh)")
ax.set_title(f"{C_GATE}, seed {SEED0}: a crossing pair the gate must refuse")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()


## 5. Known result

Notation. A record's **state leaves** are
$L = (\texttt{bids}, \texttt{asks}, \texttt{trades}, \texttt{iceberg\_meta}, \texttt{n\_active},
\texttt{ts\_counter}, \texttt{book\_full}, \texttt{truncated})$. Two states are *structurally equal*,
$S \equiv S'$, when every leaf is equal (arrays: same shape, same dtype, same bytes). $R$ is the
**fresh state**: the final state of the empty-tape run, in which no submission happens, so it is the
state the engine starts from (`book.init()`, reached through the record builder; see the
under-specification note at the end of this section). $O = (o_1, \dots, o_n)$ are the order ids of a
tape's real rows (`order_id` $\ge 1$), in tape order; pad rows have `order_id` $= -1$.

**(E0) The fresh state is empty and reproducible.**
No resting order on either side, $\#\text{trade rows used} = 0$, $\texttt{n\_active} = 0$, no live
iceberg row, $\texttt{book\_full} = \texttt{truncated} = \text{False}$; and
$R_{\text{eager}} \equiv R_{\text{jit}} \equiv R'_{\text{jit}}$ (a second, independent fresh build).

**(E1) Empty tape.** $n_{\text{sub}} = 0$; accepted, rejections and fills all empty; no flag raised;
final state $\equiv R$.

**(E2) Pad-only tape (64 sentinel rows) — simulator artefact, no spec section.**
*Record builder:* $n_{\text{sub}} = 0$; accepted, rejections and fills empty; no flag; final
$\equiv R$. *Production pad path* (the code that actually consumes pad rows in an episode, wired in
section 6): all 64 rows reach it; orders submitted $= 0$; fills $= 0$; post-state $\equiv$ its own
fresh state, which is itself empty (E0's emptiness predicates); simulated clock advance
$= \text{clock}_{\text{after}} - \text{clock}_{\text{before}} = 0$.

**(E3) The status gate, for each** $s \in \{\text{PENDING}, \text{SUSPENDED}, \text{CLOSED}\}$
(§3.1, §5.4, §3.3 phase 2), on each seed's tape:
- $n_{\text{sub}} = n$ (every real row is offered to the engine) and the engine reports
  $\text{accepted} = \text{traded} = \text{False}$ and traded quantity $0$ for each of the $n$;
- accepted $= \emptyset$; the rejection list is exactly $O$, in order, each with reason
  `contract_not_open` (refused by the status gate, not by the validator: the OPEN control below
  shows the same rows pass validation);
- fills $= \emptyset$; filled, still-resting, hidden-remaining, evicted and cancelled-or-expired
  are all empty (or all zero) — nothing was accepted, so nothing can have a fate;
- **every** post-submission state $\equiv R$ and the final state $\equiv R$ — in particular
  $\texttt{ts\_counter}$ is unchanged (§5.2: a rejected order never enters the SOB, so it receives no
  timestamp) and $\texttt{n\_active} = 0$;
- no flag raised (`book_full`, `truncated`, `trade_buffer_saturated` all False).

**(E4) The collapse, observed.** The PENDING, SUSPENDED and CLOSED records of a seed are equal on
every integer field (`compare_records` $= ()$). This certifies the single `!= OPEN` rejection as
observed behaviour; it is conformant for submission (§3.3) and does not express resting-order
survival (context quote, not certified).

**(E5) Non-vacuity control: the same tape on an OPEN contract.** Accepted ids $= O$, rejections
$= \emptyset$, and at least one fill, because the pair crosses:
$p_{\text{buy}} \ge p_{\text{sell}}$ from the draw's own parameters (`rest_side`, `p_rest_int`,
`p_agg_int`; §6.1's match condition). Nothing else about the trade is asserted here: its price and
quantity are C02's claim.

Every value above is an integer, a boolean or an id: no euro figure is asserted. The expectation is
computed from the tape's columns and `draw.params`, never from `draw.expected`.

**Under-specification notes (K2).**
1. *Battery I9 vs the known result.* I9 requires at least one accepted order per record, while C00's
   known result is that nothing is accepted. For the empty, pad-only and gated records, I9's
   `no order was accepted` outcome **is** the expectation, so section 7 records that I9 row as
   `not_applicable` (reason stated), asserts the zero-acceptance as an oracle check instead, and
   carries non-vacuity with the OPEN control (E5), on which the full battery, I9 included
   (`expects_trade=True`), must pass.
2. *Pad rows and the record builder.* The record builder skips pad rows by design (its docstring:
   "pad rows are skipped wherever they sit"), so the builder-level half of (E2) cannot fail on a
   pad row that the engine-facing path would mishandle — an inert check if it stood alone (plan rule
   3). (E2)'s production-pad-path half is therefore required, and it must show that the 64 rows
   actually **reached** the code under test.
3. *`book.init()`.* The fresh-state reference is reached through the record builder (the empty-tape
   run), because the engine's own initialiser lives in a file the K2 author may not read. Section 6
   may additionally compare $R$ against the engine's public initialiser directly, if one exists.

In [ ]:
BUY, SELL = 0, 1
FLAG_NAMES = ("book_full", "truncated", "trade_buffer_saturated")


def real_oids(draw):
    """O: the real rows' order ids (order_id >= 1), in tape order."""
    return [int(o) for o in draw.tape[:, synth.COL_OID] if int(o) >= 1]


def crosses(draw):
    """§6.1 match condition from the draw's own parameters: p_buy >= p_sell."""
    p = draw.params
    p_buy = int(p["p_rest_int"]) if int(p["rest_side"]) == BUY else int(p["p_agg_int"])
    p_sell = int(p["p_agg_int"]) if int(p["rest_side"]) == BUY else int(p["p_rest_int"])
    return p_buy >= p_sell


def expect_inert(draw):
    """(E1) / (E2) builder half: nothing is submitted, nothing changes."""
    return {"n_submissions": 0, "accepted": [], "rejected_oids": [], "n_fills": 0,
            "flags_raised": [], "final_equals_fresh": True}


def expect_pad_path(draw):
    """(E2) production half."""
    return {"rows_consumed": int(np.sum(draw.tape[:, synth.COL_OID] <= 0)), "orders_submitted": 0,
            "fills": 0, "book_equals_its_fresh": True, "fresh_is_empty": True, "clock_advance": 0}


def expect_gated(draw, status):
    """(E3) for one non-OPEN status."""
    oids = real_oids(draw)
    return {"n_submissions": len(oids), "engine_accepted": [False] * len(oids),
            "engine_traded": [False] * len(oids), "accepted": [], "rejected_oids": oids,
            "rejection_reasons": ["contract_not_open"] * len(oids), "n_fills": 0,
            "fates_empty": True, "every_post_equals_fresh": True, "final_equals_fresh": True,
            "n_active": 0, "flags_raised": []}


def expect_open_control(draw):
    """(E5): the same rows pass validation on an OPEN contract, and the pair trades."""
    return {"accepted": real_oids(draw), "rejected_oids": [], "expects_trade": crosses(draw)}


print("(E1) empty:", expect_inert(DRAW_EMPTY))
print("(E2) pad-only, builder:", expect_inert(DRAW_PAD))
print("(E2) pad-only, production path:", expect_pad_path(DRAW_PAD))
for s, d in DRAWN.items():
    e = expect_gated(d, STATUSES[0])
    c = expect_open_control(d)
    print(f"seed {s}: gated (each of {', '.join(x.name for x in STATUSES)}): {e['n_submissions']} submissions, "
          f"all rejected (oids {e['rejected_oids']}), 0 fills, every state == fresh | "
          f"OPEN control: accept {c['accepted']}, trade expected {c['expects_trade']}")


## 6. Run

### Eager and jitted (wired at K3)

> **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly these three names. Until they are defined it records a failing `section 6 not wired` check and the notebook ends `NOT CERTIFIED`.

1. **`run_seedless(name) -> (eager, jitted)`**, `name` in `("empty", "pad_only")`: the two `RunRecord`s of `DRAW_EMPTY` / `DRAW_PAD` through the record builder (`tests.omiecid.reference.omie_invariants.record_from_draw(draw, jit=False)` and `jit=True`), on an OPEN contract. Each call must build afresh (section 7 calls `run_seedless("empty")` twice to check reproducibility).
2. **`run_gated(seed, status) -> (eager, jitted)`**: the two `RunRecord`s of seed `seed`'s `C00.status_gate` tape (`DRAWN[seed]`, or `synth.CONSTRUCTIONS[C_GATE](seed)` beyond it) with `contract_status=status`, for `status` in `STATUSES` and for `OPEN` (the control).
3. **`PAD_PATH`**: a dict `{"eager": {...}, "jit": {...}}`, the 64-row pad-only tape driven through the **production code path that consumes pad rows** in an episode (the path the loader comment names for the sentinel; the record builder skips pad rows and does not count). Each mode's dict has exactly these keys:
   - `"path"`: `str`, the production entry point exercised (recorded as evidence);
   - `"rows_consumed"`: `int`, pad rows that reached that path (must be 64: the non-vacuity of this check);
   - `"fresh_book"`, `"book"`: the book state before and after the pad rows, as `tests.omiecid.reference.omie_invariants.BookSnapshot` instances (section 7 reads their leaves `bids`, `asks`, `trades`, `iceberg_meta`, `n_active`, `ts_counter`, `book_full`, `truncated` and their `resting` / `trade_rows_used` readers);
   - `"clock_before"`, `"clock_after"`: `int`, the episode's simulated clock in the path's own integer unit;
   - `"orders_submitted"`, `"fills"`: `int`, as counted by that path.

If the production pad path cannot be driven without editing locked or forbidden code, that is a `SCOPE-CHANGE` candidate, not a licence to drop (E2)'s production half.

**How K3 wired it.** `run_seedless` and `run_gated` call the record builder twice per call (eager, then jitted), so every call builds afresh. `PAD_PATH` drives the 64-row pad-only tape through `Session.step_env` with `AgentOrder.none()` on every step, under the default environment configuration (`session_clock="span_scaled"`, `orders_per_step=4`, so 16 steps). Each step reads its slice of the tape at `tape_cursor`, replays it through `_replay_orders` (which calls `_submit_order` on every row: there is no pad filter) and moves the clock with `Session._slice_time_minutes`. The clock is `state.time`, in integer minutes from GOT. `rows_consumed` counts the pad rows the steps' slices actually read; `orders_submitted` counts the order ids that entered the book (book sides and trade tape) plus agent submissions (none); `fills` counts the trade-tape rows that changed across a step.

**Not covered by this half.** Under `session_clock="got_relative"` (the real-data session clock), the first slice made only of pad rows sets the clock to GCT by design (tape exhaustion ends the episode). An all-pad tape driven that way would show a clock advance of `gct_minutes_from_got`. That case is logged as a SCOPE-CHANGE candidate in `sprint-status.md`; it is not run here.


In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3). Defines run_seedless, run_gated and PAD_PATH, the three names section 7
# consumes (contract in the markdown above). Sections 1-5 and 7 are locked (rule 2).
# ---------------------------------------------------------------------------------------
import contextlib

import jax
import jax.numpy as jnp

from omiecid_simulator.omiecid.contract import Contract
from omiecid_simulator.omiecid.contract_book import ContractBook
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from omiecid_simulator.omiecid.omie_session import AgentOrder, Session
from tests.omiecid.reference.omie_invariants import _snapshot, record_from_draw

_STATE_LEAVES = ("bids", "asks", "trades", "iceberg_meta", "n_active", "ts_counter", "book_full", "truncated")
_SEEDLESS = {"empty": lambda: synth.CONSTRUCTIONS[C_EMPTY](None),
             "pad_only": lambda: synth.CONSTRUCTIONS[C_PAD](None)}


def _both(draw, **kw):
    """(eager, jitted) RunRecords of one draw through the record builder; every call builds afresh."""
    return record_from_draw(draw, jit=False, **kw), record_from_draw(draw, jit=True, **kw)


def run_seedless(name):
    """(E0)-(E2) builder half: the empty / pad-only tape on an OPEN contract."""
    return _both(_SEEDLESS[name](), contract_status=OPEN)


def run_gated(seed, status):
    """(E3)-(E5): seed `seed`'s status-gate tape on a contract whose status is `status`."""
    draw = DRAWN[seed] if seed in DRAWN else synth.CONSTRUCTIONS[C_GATE](seed)
    return _both(draw, contract_status=status)


# ---- (E2) the production pad path -----------------------------------------------------
# The episode path that consumes pad rows (the one the loader's PAD_ORDER_ROW comment names):
# Session.step_env reads a fixed orders_per_step slice of the carried tape at tape_cursor
# (lax.dynamic_slice_in_dim, start clamped), replays it through _replay_orders, which calls
# _submit_order on EVERY row (there is no pad filter, so a pad row is inert only if the engine
# itself ignores it), and then moves the simulated clock with Session._slice_time_minutes.
# The participant's order is AgentOrder.none() on every step, so whatever the book shows comes from
# the pad rows alone. Default env configuration: session_clock="span_scaled", orders_per_step=4.
# The clock is OMIEContractState.time (integer minutes from GOT).
#
# How the path's counts are taken:
#   rows_consumed    tape positions that a step's (clamped) slice actually read and that hold a pad row;
#   orders_submitted orders that entered the book: positive order ids visible after the path (book
#                    sides, both trade-tape OID columns) and absent from the fresh book, plus agent
#                    submissions (0: AgentOrder.none());
#   fills            trade-tape rows that changed across a step (the env's own whole-row-diff idiom).
PAD_PATH_NAME = "Session.step_env (AgentOrder.none()) -> _replay_orders + Session._slice_time_minutes"
_SIDE_OID = 2          # OrderSideFeat.OID
_TRADE_OIDS = (2, 3)   # TradesFeat.PASS_OID, TradesFeat.AGRS_OID


def _visible_oids(snap):
    out = set()
    for side in (snap.bids, snap.asks):
        a = np.asarray(side)
        out |= {int(x) for x in a[:, _SIDE_OID] if int(x) > 0}
    t = np.asarray(snap.trades)
    out |= {int(x) for x in t[:, list(_TRADE_OIDS)].ravel() if int(x) > 0}
    return out


def _drive_pad_path(tape, mode):
    env = Session({"tapes": np.asarray(tape, dtype=np.int32)[None]})
    ops, n_rows = int(env.orders_per_step), int(tape.shape[0])
    n_steps = -(-n_rows // ops)
    key = jax.random.PRNGKey(0)
    noop = AgentOrder.none()
    with jax.disable_jit() if mode == "eager" else contextlib.nullcontext():
        state = env.reset(key)
        fresh, clock_before = _snapshot(state.book), int(state.time)
        reached, fills, prev = set(), 0, fresh
        for _ in range(n_steps):
            start = max(0, min(int(state.tape_cursor), n_rows - ops))      # the dynamic_slice clamp
            reached |= set(range(start, start + ops))
            state, _, _ = env.step_env(key, state, noop)
            snap = _snapshot(state.book)
            fills += int(np.sum(np.any(np.asarray(prev.trades) != np.asarray(snap.trades), axis=1)))
            prev = snap
        book, clock_after = _snapshot(state.book), int(state.time)
    return {"path": PAD_PATH_NAME,
            "rows_consumed": sum(1 for i in reached if int(tape[i, synth.COL_OID]) <= 0),
            "fresh_book": fresh, "book": book, "clock_before": clock_before, "clock_after": clock_after,
            "orders_submitted": len(_visible_oids(book) - _visible_oids(fresh)) + 0,   # + participant submissions (none)
            "fills": fills, "steps": n_steps, "cursor_after": int(state.tape_cursor)}


PAD_PATH = {m: _drive_pad_path(DRAW_PAD.tape, m) for m in ("eager", "jit")}
for m, ev in PAD_PATH.items():
    print(f"pad path [{m}]: {ev['path']}: {ev['steps']} steps, cursor 0 -> {ev['cursor_after']}, "
          f"pad rows reached {ev['rows_consumed']}/{DRAW_PAD.tape.shape[0]}; orders submitted "
          f"{ev['orders_submitted']}, fills {ev['fills']}; clock {ev['clock_before']} -> {ev['clock_after']} min")


# Under-specification note 3: the empty-tape final state R against the engine's public
# initialiser, directly. Shown here, not asserted (section 7 owns the checks).
def _differing(a, b):
    return [k for k in _STATE_LEAVES if not (np.asarray(getattr(a, k)).dtype == np.asarray(getattr(b, k)).dtype
                                            and np.array_equal(np.asarray(getattr(a, k)), np.asarray(getattr(b, k))))]


_R_probe = run_seedless("empty")[1].final_snapshot
_init = _snapshot(ContractBook(
    Contract(contract_id=0, delivery_start_minutes=555, delivery_end_minutes=570, gct_minutes=495,
             status=OPEN), CONFIG, OMIEMarketConfig()).init())
print("ContractBook.init() vs R, leaves that differ:", _differing(_init, _R_probe) or "none")
print("pad path fresh book (Session.reset) vs R, leaves that differ:",
      _differing(PAD_PATH["jit"]["fresh_book"], _R_probe) or "none")


## 7. Certification

The oracle comparison in integer ticks, ids and booleans, then the invariant battery (RB, I1–I9) folded per record as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

Battery policy (section 5, note 1): on the empty, pad-only and gated records the I9 row is recorded `not_applicable` **only** when its sole violation is "no order was accepted", which is C00's expectation; any other I9 outcome is kept. The OPEN control runs the full battery with `expects_trade=True`.

In [ ]:
import dataclasses
import traceback

from tests.omiecid.reference.omie_invariants import BatteryReport, compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Inertness and the contract-status gate",
    known_result="zero trades; state == fresh init leaf for leaf; every submission rejected; "
                 "n_active = 0; no flags raised",
    oracle_kind="exact-literal", quotes=QUOTES, citations=CITATIONS, constructions=CONSTRUCTIONS,
    requested_draws=REQUESTED_RECORDS, runtime_forecast_s=40.0, started_at=T0,
    downstream="every later scenario and all of Track B: pad-row semantics fix every episode length and "
               "reward, and the status gate underlies C11 (expiry at gate closure) and C16 (multi-contract "
               "isolation)",
)

LEAVES = ("bids", "asks", "trades", "iceberg_meta", "n_active", "ts_counter", "book_full", "truncated")
R_GATE, R_PAD, R_STATE = "§3.1, §5.4, §3.3", "simulator artefact (no spec section)", "§5.2"


def leaf_diff(a, b):
    """Names of the state leaves on which a and b differ (arrays: shape, dtype and bytes)."""
    out = []
    for k in LEAVES:
        x, y = getattr(a, k, None), getattr(b, k, None)
        if isinstance(x, np.ndarray) or isinstance(y, np.ndarray):
            x, y = np.asarray(x), np.asarray(y)
            same = x.shape == y.shape and x.dtype == y.dtype and np.array_equal(x, y)
        else:
            same = x == y
        if not same:
            out.append(k)
    return out


def emptiness(s):
    """(E0)'s emptiness predicates on one BookSnapshot: the names of those that fail."""
    bad = []
    if s.resting(BUY) or s.resting(SELL):
        bad.append("a resting order is live")
    if s.trade_rows_used() != 0:
        bad.append("trade rows used")
    if int(s.n_active) != 0:
        bad.append("n_active != 0")
    if np.asarray(s.iceberg_meta).size != 0:
        bad.append("live iceberg rows")
    if bool(s.book_full) or bool(s.truncated):
        bad.append("flag raised")
    return bad


def flags_raised(rec):
    return sorted(k for k, v in dict(rec.flags).items() if bool(v))


def fates_empty(rec):
    maps = (rec.filled, rec.still_resting, rec.hidden_remaining, rec.evicted)
    return all(all(int(v) == 0 for v in dict(m).values()) for m in maps) and \
        all(len(v) == 0 for v in dict(rec.cancelled_or_expired).values())


def battery(rec, *, expects_trade, inert):
    rep = run_battery(rec, expects_trade=expects_trade)
    if not inert:
        return rep
    out = []
    for r in rep.results:
        if (r.check_id == "I9" and r.status == "fail"
                and {v.field for v in r.violations} == {"accepted"}):
            r = dataclasses.replace(r, status="not_applicable", evidence=dict(
                r.evidence, reason="C00: zero accepted orders is the expectation; asserted as an oracle check, "
                                   "non-vacuity carried by the OPEN control"))
        out.append(r)
    return BatteryReport(results=tuple(out))


def guarded(label, fn, *args):
    """Call a section-6 function; a raise is recorded as a failed check, never propagated."""
    try:
        return fn(*args)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, detail=f"{type(exc).__name__}: {exc} | "
                   + traceback.format_exc().splitlines()[-1], rule=R_GATE)
        return None


def eager_jit(label, seed, eager, jitted):
    diff = compare_records(eager, jitted)
    cert.check(f"{label}: eager == jitted", diff == (), kind="eager_jit", seed=seed, expected=True,
               observed=diff == (), unit="bool", detail=", ".join(diff))


def check_inert_record(label, seed, rec, R, e, rule):
    cert.check_equal(f"{label}: submissions", rec.n_submissions, e["n_submissions"], seed=seed, unit="count", rule=rule)
    cert.check_equal(f"{label}: accepted ids", sorted(rec.accepted), e["accepted"], seed=seed, unit="other", rule=rule)
    cert.check_equal(f"{label}: rejected ids", [r.oid for r in rec.rejections], e["rejected_oids"], seed=seed,
                     unit="other", rule=rule)
    cert.check_equal(f"{label}: fills", len(rec.fills), e["n_fills"], seed=seed, unit="count", rule=rule)
    cert.check_equal(f"{label}: flags raised", flags_raised(rec), e["flags_raised"], seed=seed, unit="other", rule=rule)
    d = leaf_diff(rec.final_snapshot, R)
    cert.check(f"{label}: final state == fresh (leaf for leaf)", d == [], seed=seed, expected=[], observed=d,
               unit="other", detail=", ".join(d), rule=rule)


wired = callable(run_seedless) and callable(run_gated) and isinstance(PAD_PATH, dict)
cert.check("section 6 wired (run_seedless, run_gated, PAD_PATH defined)", wired, expected=True, observed=wired,
           unit="bool", detail="" if wired else "section 6 is still the K2 stub", rule=R_GATE)

R = None
if wired:
    # ---- (E0) the fresh state, and (E1) the empty tape ---------------------------------
    pair = guarded("empty", run_seedless, "empty")
    again = guarded("empty (second build)", run_seedless, "empty")
    if pair is not None:
        eager, jitted = pair
        R = jitted.final_snapshot
        eager_jit("empty", None, eager, jitted)
        bad = emptiness(R)
        cert.check("(E0) fresh state is empty", bad == [], expected=[], observed=bad, unit="other",
                   detail="; ".join(bad), rule=R_STATE)
        d = leaf_diff(eager.final_snapshot, R)
        cert.check("(E0) fresh state reproducible: eager == jitted", d == [], expected=[], observed=d,
                   unit="other", detail=", ".join(d), rule=R_STATE)
        if again is not None:
            d = leaf_diff(again[1].final_snapshot, R)
            cert.check("(E0) fresh state reproducible: second build", d == [], expected=[], observed=d,
                       unit="other", detail=", ".join(d), rule=R_STATE)
        check_inert_record("(E1) empty", None, jitted, R, expect_inert(DRAW_EMPTY), R_STATE)
        cert.add_battery(battery(jitted, expects_trade=False, inert=True), seed=None)
        cert.record_capacity({**jitted.observed_capacity, "hops": None})

    # ---- (E2) pad-only: record builder half -------------------------------------------
    pair = guarded("pad-only", run_seedless, "pad_only")
    if pair is not None and R is not None:
        eager, jitted = pair
        eager_jit("pad-only", None, eager, jitted)
        check_inert_record("(E2) pad-only (builder)", None, jitted, R, expect_inert(DRAW_PAD), R_PAD)
        cert.add_battery(battery(jitted, expects_trade=False, inert=True), seed=None)
        cert.record_capacity({**jitted.observed_capacity, "hops": None})

    # ---- (E2) pad-only: production pad path -------------------------------------------
    ep = expect_pad_path(DRAW_PAD)
    for mode in ("eager", "jit"):
        ev = PAD_PATH.get(mode)
        if not isinstance(ev, dict):
            cert.check(f"(E2) pad path [{mode}] evidence present", False, expected=True, observed=False,
                       unit="bool", rule=R_PAD)
            continue
        lab = f"(E2) pad path [{mode}] ({ev.get('path', '?')})"
        cert.check_equal(f"{lab}: rows consumed", ev.get("rows_consumed"), ep["rows_consumed"], unit="count", rule=R_PAD)
        cert.check_equal(f"{lab}: orders submitted", ev.get("orders_submitted"), ep["orders_submitted"],
                         unit="count", rule=R_PAD)
        cert.check_equal(f"{lab}: fills", ev.get("fills"), ep["fills"], unit="count", rule=R_PAD)
        try:
            d = leaf_diff(ev["book"], ev["fresh_book"])
            bad = emptiness(ev["fresh_book"])
        except Exception as exc:  # noqa: BLE001
            d, bad = [f"unreadable: {type(exc).__name__}"], [f"unreadable: {type(exc).__name__}"]
        cert.check(f"{lab}: post-state == its fresh state", d == [], expected=[], observed=d, unit="other",
                   detail=", ".join(d), rule=R_PAD)
        cert.check(f"{lab}: its fresh state is empty", bad == [], expected=[], observed=bad, unit="other",
                   detail="; ".join(bad), rule=R_PAD)
        try:
            adv = int(ev["clock_after"]) - int(ev["clock_before"])
        except Exception as exc:  # noqa: BLE001
            adv = f"unreadable: {type(exc).__name__}"
        cert.check_equal(f"{lab}: clock advance", adv, ep["clock_advance"], unit="count", rule=R_PAD)
    if isinstance(PAD_PATH.get("eager"), dict) and isinstance(PAD_PATH.get("jit"), dict):
        keys = ("rows_consumed", "orders_submitted", "fills", "clock_before", "clock_after")
        same = all(PAD_PATH["eager"].get(k) == PAD_PATH["jit"].get(k) for k in keys)
        try:
            same = same and leaf_diff(PAD_PATH["eager"]["book"], PAD_PATH["jit"]["book"]) == []
        except Exception:  # noqa: BLE001
            same = False
        cert.check("(E2) pad path: eager == jitted", same, kind="eager_jit", expected=True, observed=same, unit="bool")

    # ---- (E3)-(E5) the status gate, seed by seed ---------------------------------------
    valid_seeds, seed = 0, SEED0
    while R is not None and valid_seeds < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
        draw = DRAWN.get(seed) or synth.CONSTRUCTIONS[C_GATE](seed)
        runs = {st: guarded(f"seed {seed} {st.name}", run_gated, seed, st) for st in (*STATUSES, OPEN)}
        if any(v is None for v in runs.values()):
            seed += 1
            continue
        ec = expect_open_control(draw)
        reports = {st: battery(runs[st][1], expects_trade=(st == OPEN and ec["expects_trade"]), inert=(st != OPEN))
                   for st in runs}
        if not all(r.valid for r in reports.values()):
            cert.discard(seed, "I8 invalid: a record of this seed exceeds the capacity envelope")
            seed += 1
            continue
        valid_seeds += 1
        cert.record_case(seed, draw.tape, draw.params)
        for st, rep in reports.items():
            cert.add_battery(rep, seed=seed)
            cert.record_capacity({**runs[st][1].observed_capacity, "hops": None})
            eager_jit(f"seed {seed} {st.name}", seed, *runs[st])

        for st in STATUSES:
            rec, e, lab = runs[st][1], expect_gated(draw, st), f"seed {seed} {st.name}"
            check_inert_record(f"(E3) {lab}", seed, rec, R, e, R_GATE)
            cert.check_equal(f"(E3) {lab}: engine accepted", [bool(r.accepted) for r in rec.engine_reports],
                             e["engine_accepted"], seed=seed, unit="other", rule=R_GATE)
            cert.check_equal(f"(E3) {lab}: engine traded", [bool(r.traded) for r in rec.engine_reports],
                             e["engine_traded"], seed=seed, unit="other", rule=R_GATE)
            qty = [int(round(float(r.trade_qty_mw) / float(CONFIG.qty_tick))) for r in rec.engine_reports]
            cert.check_equal(f"(E3) {lab}: engine traded qty_int", qty, [0] * len(e["engine_accepted"]),
                             seed=seed, unit="qty_int", rule=R_GATE)
            cert.check_equal(f"(E3) {lab}: rejection reasons", [r.reason for r in rec.rejections],
                             e["rejection_reasons"], seed=seed, unit="other", rule=R_GATE)
            cert.check_equal(f"(E3) {lab}: order fates all empty", fates_empty(rec), e["fates_empty"], seed=seed,
                             unit="bool", rule=R_GATE)
            posts = [leaf_diff(s, R) for s in rec.post_snapshots]
            bad = [i for i, d in enumerate(posts) if d]
            cert.check(f"(E3) {lab}: every post-submission state == fresh", bad == [], seed=seed, expected=[],
                       observed=bad, unit="other",
                       detail="; ".join(f"submission {i}: {', '.join(posts[i])}" for i in bad), rule=R_STATE)
            cert.check_equal(f"(E3) {lab}: n_active", int(rec.final_snapshot.n_active), e["n_active"], seed=seed,
                             unit="count", rule=R_GATE)

        # (E4) the collapse, observed
        base = runs[STATUSES[0]][1]
        for st in STATUSES[1:]:
            diff = compare_records(base, runs[st][1])
            cert.check(f"(E4) seed {seed}: {st.name} record == {STATUSES[0].name} record", diff == (), seed=seed,
                       expected=[], observed=list(diff), unit="other", detail=", ".join(diff), rule="§9.2")

        # (E5) the OPEN control
        rec = runs[OPEN][1]
        cert.check_equal(f"(E5) seed {seed} OPEN: accepted ids", sorted(rec.accepted), sorted(ec["accepted"]),
                         seed=seed, unit="other", rule="§5.4")
        cert.check_equal(f"(E5) seed {seed} OPEN: rejected ids", [r.oid for r in rec.rejections], ec["rejected_oids"],
                         seed=seed, unit="other", rule="§5.4")
        cert.check_equal(f"(E5) seed {seed} OPEN: at least one fill", len(rec.fills) >= 1, ec["expects_trade"],
                         seed=seed, unit="bool", rule="§6.1")
        seed += 1

display(Markdown(cert.table()))


## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")


In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))
